# AnimalCLEF2026 Masked Dual-Foundation Search v20260425

This notebook keeps the strongest parts of the current best public-LB pipeline and upgrades the descriptor stage instead of rewriting clustering from scratch.

Core idea:

1. Keep the masked-crops plus original-image fusion that already proved itself, but score candidate profiles against both the old 0.27429 baseline regime and the newer 0.27632 ARWild v3_update regime.
2. Extract multiple strong wildlife descriptors sequentially and cache them:
   - `conservationxlabs/miewid-msv3`
   - `BVRA/MegaDescriptor-L-384` with ImageNet normalization (baseline-compatible)
   - `BVRA/MegaDescriptor-L-384` with model-card `0.5 / 0.5 / 0.5` normalization
3. Build species-aware dual-foundation and tri-foundation candidates.
4. Tune thresholds on train identities for labeled species and use a transparent proxy to pick the default `submission.csv` plus a named selected artifact.

Outputs:

- `/kaggle/working/submission.csv`
- `/kaggle/working/submission_selected_<profile>.csv`
- `/kaggle/working/dualfoundation_candidate_report_v20260425.csv`
- `/kaggle/working/dualfoundation_profile_summary_v20260425.csv`
- `/kaggle/working/dualfoundation_selected_profile_v20260425.json`
- `/kaggle/working/animalclef_dualfoundation_search_v20260425/submissions/submission_<profile>.csv`
        

In [ ]:
!pip install -q timm transformers wildlife-datasets wildlife-tools scikit-learn opencv-python-headless
        

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFile

import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

import timm
from transformers import AutoModel
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import normalize
from sklearn.cluster import AgglomerativeClustering
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")
        

In [ ]:
# =========================
# Config
# =========================

VERSION = "masked_dualfoundation_search_v20260425"
SEED = 20260425
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
print("GPU:", GPU_NAME)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT_DIR = WORK_DIR / "animalclef_dualfoundation_search_v20260425"
CACHE_DIR = OUT_DIR / "cache"
SUB_DIR = OUT_DIR / "submissions"
REPORT_DIR = OUT_DIR / "reports"
for d in [OUT_DIR, CACHE_DIR, SUB_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

USE_MASKED_CROPS = True
FALLBACK_TO_ORIGINAL_WHEN_MASK_MISSING = True
TTA_VARIANTS_ALL = ["identity", "hflip", "contrast"]
# Kaggle/Jupyter can throw worker teardown assertions on DataLoader cleanup.
# Default to single-process loading for stability; override with DUALFOUNDATION_NUM_WORKERS if needed.
NUM_WORKERS = int(os.environ.get("DUALFOUNDATION_NUM_WORKERS", "0"))

if "L4" in GPU_NAME:
    BATCH_SIZE_BY_BACKBONE = {
        "miewid": 96,
        "mega_l384_oldnorm": 96,
        "mega_l384": 96,
    }
elif "P100" in GPU_NAME:
    BATCH_SIZE_BY_BACKBONE = {
        "miewid": 56,
        "mega_l384_oldnorm": 64,
        "mega_l384": 64,
    }
else:
    BATCH_SIZE_BY_BACKBONE = {
        "miewid": 64,
        "mega_l384_oldnorm": 72,
        "mega_l384": 72,
    }

BACKBONE_CONFIGS = {
    "miewid": {
        "family": "miewid",
        "image_size": 512,
        "norm": "imagenet",
    },
    "mega_l384_oldnorm": {
        "family": "timm",
        "model_name": "hf-hub:BVRA/MegaDescriptor-L-384",
        "image_size": 384,
        "norm": "imagenet",
    },
    "mega_l384": {
        "family": "timm",
        "model_name": "hf-hub:BVRA/MegaDescriptor-L-384",
        "image_size": 384,
        "norm": "mega",
    },
}

SPECIES_CONFIG = {
    "LynxID2025": {
        "base_threshold": 0.47,
        "qe_k": 5,
    },
    "SalamanderID2025": {
        "base_threshold": 0.13,
        "qe_k": 3,
    },
    "SeaTurtleID2022": {
        "base_threshold": 0.49,
        "qe_k": 8,
    },
    "TexasHornedLizards": {
        "base_threshold": 0.30,
        "qe_k": 5,
    },
}

SPECIES_ORDER = list(SPECIES_CONFIG.keys())

ANCHOR_REGIMES = {
    "baseline_027429": {
        "weight": 0.45,
        "target_clusters": {
            "LynxID2025": 64,
            "SalamanderID2025": 593,
            "SeaTurtleID2022": 215,
            "TexasHornedLizards": 27,
        },
        "max_cluster_ok": {
            "LynxID2025": 177,
            "SalamanderID2025": 6,
            "SeaTurtleID2022": 15,
            "TexasHornedLizards": 83,
        },
    },
    "arwild_v3_update_027632": {
        "weight": 0.55,
        "target_clusters": {
            "LynxID2025": 135,
            "SalamanderID2025": 514,
            "SeaTurtleID2022": 199,
            "TexasHornedLizards": 71,
        },
        "max_cluster_ok": {
            "LynxID2025": 67,
            "SalamanderID2025": 10,
            "SeaTurtleID2022": 9,
            "TexasHornedLizards": 26,
        },
    },
}

THRESHOLD_GRID_RADIUS_LOW = 0.10
THRESHOLD_GRID_RADIUS_HIGH = 0.15
THRESHOLD_GRID_STEPS = 12
DEFAULT_PROFILE_HINT = None
        

In [ ]:
# =========================
# Data root + masked crop discovery
# =========================

def find_competition_root():
    candidates = [
        Path("/kaggle/input/animal-clef-2026"),
        Path("/kaggle/input/competitions/animal-clef-2026"),
        Path("/root/.cache/kagglehub/competitions/animal-clef-2026"),
        Path.cwd() / "animal-clef-2026",
    ]
    for root in candidates:
        if (root / "metadata.csv").exists() and (root / "sample_submission.csv").exists():
            return root

    for base in [Path("/kaggle/input"), Path("/content"), Path.cwd()]:
        if base.exists():
            for p in base.rglob("metadata.csv"):
                if (p.parent / "sample_submission.csv").exists():
                    return p.parent

    raise FileNotFoundError("Could not find AnimalCLEF2026 data root.")


DATA_ROOT = find_competition_root()
print("DATA_ROOT:", DATA_ROOT)

metadata = pd.read_csv(DATA_ROOT / "metadata.csv").reset_index(drop=True)
sample_submission = pd.read_csv(DATA_ROOT / "sample_submission.csv")
metadata["row_idx"] = np.arange(len(metadata))
metadata["abs_path"] = metadata["path"].map(
    lambda p: str((DATA_ROOT / str(p)).resolve()) if not Path(str(p)).is_absolute() else str(p)
)

if "dataset" not in metadata.columns and "species" in metadata.columns:
    metadata["dataset"] = metadata["species"]

print(metadata.groupby(["dataset", "split"]).size())
print("sample submission rows:", len(sample_submission))


def find_sam_view_manifests():
    manifests = []

    direct_candidates = [
        WORK_DIR / "animalclef_sam3_views_cache" / "reports" / "view_manifest_sam3_all_species.csv",
        Path.cwd() / "animalclef_sam3_views_cache" / "reports" / "view_manifest_sam3_all_species.csv",
    ]

    for manifest in direct_candidates:
        if manifest.exists():
            manifests.append(manifest)

    for base in [Path("/kaggle/input"), WORK_DIR, Path.cwd()]:
        if base.exists():
            try:
                for p in base.rglob("view_manifest_sam3_all_species.csv"):
                    if p.exists():
                        manifests.append(p)
            except Exception:
                pass

    unique = []
    seen = set()
    for manifest in manifests:
        rp = str(manifest.resolve())
        if rp not in seen:
            seen.add(rp)
            unique.append(manifest)
    return unique


def load_best_sam_view_manifest():
    best_df = None
    best_manifest = None
    best_root = None
    best_score = (-1, -1)
    for manifest in SAM_VIEW_MANIFESTS:
        try:
            df = pd.read_csv(manifest)
        except Exception:
            continue
        required = {"row_idx", "mask_full_path", "loose_path", "mask_ok"}
        if not required.issubset(df.columns):
            continue
        score = (int(df["mask_ok"].fillna(False).sum()), len(df))
        if score > best_score:
            best_df = df.copy()
            best_manifest = manifest
            best_root = manifest.parent.parent if manifest.parent.name == "reports" else manifest.parent
            best_score = score
    return best_df, best_manifest, best_root


def find_masked_crop_roots():
    roots = []

    direct_candidates = [
        WORK_DIR / "animalclef_sam3_masked_reid" / "masked_crops",
        WORK_DIR / "masked_crops",
        Path.cwd() / "animalclef_sam3_masked_reid" / "masked_crops",
        Path.cwd() / "masked_crops",
    ]

    for root in direct_candidates:
        if root.exists():
            roots.append(root)

    for base in [Path("/kaggle/input"), WORK_DIR, Path.cwd()]:
        if base.exists():
            try:
                for p in base.rglob("masked_crops"):
                    if p.exists():
                        roots.append(p)
            except Exception:
                pass

    unique_roots = []
    seen = set()
    for r in roots:
        rp = str(r.resolve())
        if rp not in seen:
            seen.add(rp)
            unique_roots.append(r)
    return unique_roots

SAM_VIEW_MANIFESTS = find_sam_view_manifests()
SAM_VIEW_DF, SAM_VIEW_MANIFEST_PATH, SAM_VIEW_EXPORT_ROOT = load_best_sam_view_manifest()
print("Detected SAM view manifests:")
for p in SAM_VIEW_MANIFESTS:
    print(" -", p)
if SAM_VIEW_MANIFEST_PATH is not None:
    print("[sam views] using", SAM_VIEW_MANIFEST_PATH)
    print("[sam views] export root", SAM_VIEW_EXPORT_ROOT)

MASKED_CROP_ROOTS = find_masked_crop_roots()
print("Detected masked crop roots:")
for r in MASKED_CROP_ROOTS:
    print(" -", r)

if USE_MASKED_CROPS and SAM_VIEW_DF is None and len(MASKED_CROP_ROOTS) == 0:
    print("WARNING: No masked_crops folder detected. Falling back to original images.")


def possible_mask_names(original_path):
    p = Path(original_path)
    stem = p.stem
    return [
        f"{stem}_masked.jpg",
        f"{stem}_masked.png",
        f"{stem}.jpg",
        f"{stem}.png",
        p.name,
    ]


def resolve_masked_crop(original_path, species_name):
    for root in MASKED_CROP_ROOTS:
        for name in possible_mask_names(original_path):
            candidate = root / species_name / name
            if candidate.exists():
                return str(candidate), True
        for name in possible_mask_names(original_path):
            matches = list(root.rglob(name))
            if matches:
                return str(matches[0]), True

    if FALLBACK_TO_ORIGINAL_WHEN_MASK_MISSING:
        return original_path, False
    raise FileNotFoundError(f"Masked crop not found for {original_path}")


def resolve_sam_manifest_mask(path_value, export_root, species_name, original_path):
    if isinstance(path_value, str) and Path(path_value).exists():
        return str(Path(path_value)), True
    stem = Path(original_path).stem
    candidate = Path(export_root) / "mask_full_canvas" / species_name / f"{stem}_maskfull.jpg"
    if candidate.exists():
        return str(candidate), True
    if FALLBACK_TO_ORIGINAL_WHEN_MASK_MISSING:
        return original_path, False
    raise FileNotFoundError(f"SAM manifest mask not found for {original_path}")


if USE_MASKED_CROPS and SAM_VIEW_DF is not None:
    view_meta = SAM_VIEW_DF[["row_idx", "mask_full_path", "mask_ok"]].copy()
    view_meta["sam_export_root"] = str(SAM_VIEW_EXPORT_ROOT)
    metadata = metadata.merge(view_meta, on="row_idx", how="left")
    mapped_paths = []
    mask_found = []
    for _, row in metadata.iterrows():
        p, exists = resolve_sam_manifest_mask(
            row.get("mask_full_path"),
            row.get("sam_export_root"),
            row["dataset"],
            row["abs_path"],
        )
        mapped_paths.append(p)
        mask_found.append(bool(row.get("mask_ok", False)) and exists)
    metadata["masked_path"] = mapped_paths
    metadata["mask_found"] = mask_found
elif USE_MASKED_CROPS:
    mapped_paths = []
    mask_found = []
    for _, row in metadata.iterrows():
        p, ok = resolve_masked_crop(row["abs_path"], row["dataset"])
        mapped_paths.append(p)
        mask_found.append(ok)
    metadata["masked_path"] = mapped_paths
    metadata["mask_found"] = mask_found
else:
    metadata["masked_path"] = metadata["abs_path"]
    metadata["mask_found"] = False

print("Masked crop coverage by dataset:")
display(metadata.groupby("dataset")["mask_found"].agg(["sum", "count", "mean"]))
        

In [ ]:
# =========================
# Models + feature extraction
# =========================

def cleanup_memory(tag=""):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
        if tag:
            free_gb = torch.cuda.mem_get_info()[0] / (1024 ** 3)
            print(f"[gpu] {tag} free={free_gb:.2f}GB")


def patch_transformers_for_miewid():
    try:
        from transformers.modeling_utils import PreTrainedModel
        current_value = getattr(PreTrainedModel, "all_tied_weights_keys", None)
        if current_value is None or not hasattr(current_value, "keys"):
            PreTrainedModel.all_tied_weights_keys = {}
    except Exception as e:
        print(f"[warn] Could not apply MiewID transformers patch: {e}")


class MiewIDWrapper(nn.Module):
    def __init__(self):
        super().__init__()
        patch_transformers_for_miewid()
        self.backbone = AutoModel.from_pretrained(
            "conservationxlabs/miewid-msv3",
            trust_remote_code=True,
            low_cpu_mem_usage=False,
        )
        if not hasattr(self.backbone, "all_tied_weights_keys"):
            self.backbone.all_tied_weights_keys = {}

    def forward(self, x):
        return self.backbone(x)


class TimmEmbeddingWrapper(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
        )

    def forward(self, x):
        return self.backbone(x)


def get_model(backbone_name):
    cfg = BACKBONE_CONFIGS[backbone_name]
    family = cfg["family"]

    if family == "miewid":
        model = MiewIDWrapper()
    elif family == "timm":
        model = TimmEmbeddingWrapper(cfg["model_name"])
    else:
        raise ValueError(f"Unknown backbone family for {backbone_name}: {family}")

    model.to(DEVICE)
    model.eval()
    if torch.cuda.is_available():
        try:
            model = model.to(memory_format=torch.channels_last)
        except Exception:
            pass
    return model


def output_to_tensor(model_output):
    if isinstance(model_output, torch.Tensor):
        x = model_output
    elif isinstance(model_output, dict):
        x = None
        for key in ["embedding", "embeddings", "features", "pooler_output", "last_hidden_state"]:
            if key in model_output and model_output[key] is not None:
                x = model_output[key]
                break
        if x is None:
            raise TypeError(f"Unsupported output dict keys: {list(model_output.keys())}")
    elif hasattr(model_output, "pooler_output") and model_output.pooler_output is not None:
        x = model_output.pooler_output
    elif hasattr(model_output, "last_hidden_state"):
        x = model_output.last_hidden_state.mean(dim=1)
    elif isinstance(model_output, (tuple, list)):
        x = model_output[0]
    else:
        raise TypeError(type(model_output))

    if x.ndim == 4:
        x = torch.nn.functional.adaptive_avg_pool2d(x, 1).flatten(1)
    if x.ndim == 3:
        x = x.mean(dim=1)
    return x


class PathImageDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = list(paths)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        try:
            image = Image.open(path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (512, 512), (238, 238, 232))
        return self.transform(image)


def fixed_flip(image):
    if hasattr(Image, "Transpose"):
        return image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    return image.transpose(Image.FLIP_LEFT_RIGHT)


def fixed_contrast(image):
    image = ImageEnhance.Contrast(image).enhance(1.12)
    image = ImageEnhance.Sharpness(image).enhance(1.08)
    return image


def norm_for_backbone(backbone_name):
    norm_type = BACKBONE_CONFIGS[backbone_name]["norm"]
    if norm_type == "imagenet":
        return (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    if norm_type == "mega":
        return (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
    raise ValueError(norm_type)


def make_transform(backbone_name, variant):
    image_size = BACKBONE_CONFIGS[backbone_name]["image_size"]
    mean, std = norm_for_backbone(backbone_name)

    ops = [T.Resize((image_size, image_size))]
    if variant == "identity":
        pass
    elif variant == "hflip":
        ops.append(T.Lambda(fixed_flip))
    elif variant == "contrast":
        ops.append(T.Lambda(fixed_contrast))
    else:
        raise ValueError(f"Unknown TTA variant: {variant}")

    ops += [
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ]
    return T.Compose(ops)


def l2_normalize(x):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 1:
        x = x[None, :]
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)


def cache_key_for_paths(paths, extra_text):
    short = list(paths[:5]) + list(paths[-5:]) + [str(len(paths)), extra_text]
    joined = "|".join(map(str, short))
    return hashlib.md5(joined.encode("utf-8")).hexdigest()[:12]


def extract_one_tta(model, paths, backbone_name, cache_prefix, variant):
    key = cache_key_for_paths(paths, f"{cache_prefix}_{backbone_name}_{variant}")
    cache_path = CACHE_DIR / f"{cache_prefix}_{variant}_{key}.npy"

    if cache_path.exists():
        arr = np.load(cache_path)
        if arr.shape[0] == len(paths):
            print("[cache] loaded", cache_path.name, arr.shape)
            return l2_normalize(arr)

    dataset = PathImageDataset(paths, make_transform(backbone_name, variant))
    batch_size = BATCH_SIZE_BY_BACKBONE[backbone_name]

    while batch_size >= 1:
        try:
            loader = DataLoader(
                dataset,
                batch_size=batch_size,
                shuffle=False,
                num_workers=NUM_WORKERS,
                pin_memory=torch.cuda.is_available(),
            )

            features = []
            with torch.inference_mode():
                for batch in tqdm(loader, desc=f"{cache_prefix} | {variant} | bs={batch_size}", leave=False):
                    batch = batch.to(DEVICE, non_blocking=True)
                    if torch.cuda.is_available():
                        batch = batch.contiguous(memory_format=torch.channels_last)
                        with torch.autocast(device_type="cuda", dtype=torch.float16):
                            output = model(batch)
                    else:
                        output = model(batch)
                    feature_tensor = output_to_tensor(output)
                    features.append(feature_tensor.detach().float().cpu().numpy())

            arr = np.concatenate(features, axis=0)
            arr = l2_normalize(arr)
            np.save(cache_path, arr.astype(np.float16))
            print("[cache] wrote", cache_path.name, arr.shape)
            return arr

        except RuntimeError as e:
            if "out of memory" not in str(e).lower() or batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            cleanup_memory(f"oom {backbone_name} -> bs {batch_size}")

    raise RuntimeError(f"Failed to extract features for {backbone_name} / {variant}")


def extract_tta_bank(model, paths, backbone_name, cache_prefix, variants):
    bank = {}
    for variant in variants:
        bank[variant] = extract_one_tta(
            model=model,
            paths=paths,
            backbone_name=backbone_name,
            cache_prefix=cache_prefix,
            variant=variant,
        )
    return bank


def average_tta_bank(bank, variants):
    arrays = []
    for variant in variants:
        if variant not in bank:
            raise KeyError(f"TTA variant {variant} missing from bank. Available: {list(bank.keys())}")
        arrays.append(bank[variant])
    return l2_normalize(np.mean(np.stack(arrays, axis=0), axis=0))
        

In [ ]:
# =========================
# Clustering + candidate helpers
# =========================

def query_expansion(features, k=5, alpha=0.5):
    if len(features) <= 1:
        return features
    k = min(int(k), len(features))
    sim_matrix = np.dot(features, features.T)
    knn_indices = np.argsort(sim_matrix, axis=1)[:, -k:]
    expanded = np.zeros_like(features)
    for i in range(features.shape[0]):
        expanded[i] = features[i] + alpha * np.mean(features[knn_indices[i]], axis=0)
    return l2_normalize(expanded)


def run_agglomerative_from_features(features, threshold):
    if len(features) == 0:
        return np.array([], dtype=int)
    if len(features) == 1:
        return np.array([0], dtype=int)

    distance_matrix = np.clip(1.0 - np.dot(features, features.T), 0.0, 2.0)
    try:
        model = AgglomerativeClustering(
            n_clusters=None,
            metric="precomputed",
            linkage="average",
            distance_threshold=float(threshold),
        )
    except TypeError:
        model = AgglomerativeClustering(
            n_clusters=None,
            affinity="precomputed",
            linkage="average",
            distance_threshold=float(threshold),
        )
    return model.fit_predict(distance_matrix)


def tune_threshold_on_train(train_features, train_labels, base_threshold):
    valid_mask = pd.Series(train_labels).notna().to_numpy()
    train_features = train_features[valid_mask]
    train_labels = np.asarray(train_labels)[valid_mask]

    if len(train_labels) < 3 or len(np.unique(train_labels)) < 2:
        return float(base_threshold), []

    low = max(0.03, float(base_threshold) - THRESHOLD_GRID_RADIUS_LOW)
    high = min(0.95, float(base_threshold) + THRESHOLD_GRID_RADIUS_HIGH)
    grid = np.round(np.linspace(low, high, THRESHOLD_GRID_STEPS), 4)

    best_score = -999.0
    best_threshold = float(base_threshold)
    history = []
    for threshold in grid:
        labels = run_agglomerative_from_features(train_features, threshold)
        score = adjusted_rand_score(train_labels, labels)
        n_clusters = len(np.unique(labels))
        history.append((float(threshold), float(score), int(n_clusters)))
        if score > best_score:
            best_score = float(score)
            best_threshold = float(threshold)
    return best_threshold, history


def fuse_feature_pair(masked_features, original_features, mask_weight=0.70, original_weight=0.30, fusion_mode="concat"):
    if fusion_mode == "masked_only":
        return l2_normalize(masked_features)
    if fusion_mode == "original_only":
        return l2_normalize(original_features)
    if fusion_mode == "sum":
        return l2_normalize(mask_weight * masked_features + original_weight * original_features)
    if fusion_mode == "concat":
        return l2_normalize(np.concatenate([mask_weight * masked_features, original_weight * original_features], axis=1))
    raise ValueError(f"Unknown fusion_mode: {fusion_mode}")


def cluster_shape(part):
    counts = part["cluster"].value_counts()
    return {
        "n_test_images": int(len(part)),
        "n_clusters": int(part["cluster"].nunique()),
        "max_cluster_size": int(counts.max()) if len(counts) else 0,
        "singletons": int((counts == 1).sum()) if len(counts) else 0,
    }


def make_submission_part(features, image_ids, species, threshold):
    labels = run_agglomerative_from_features(features, threshold)
    clusters = [f"cluster_{species}_{label}" for label in labels]
    return pd.DataFrame({"image_id": image_ids, "cluster": clusters})


def get_local_indices(rows, local_index_by_global):
    return [local_index_by_global[int(x)] for x in rows["row_idx"].tolist()]


def get_species_backbones(recipe, species):
    species_backbones = recipe.get("species_backbones", {})
    default_backbones = recipe.get("default_backbones", ["mega_l384"])
    value = species_backbones.get(species, default_backbones)
    if isinstance(value, str):
        value = [value]
    return list(value)


def get_species_backbone_weights(recipe, species):
    species_weights = recipe.get("species_backbone_weights", {})
    default_weights = recipe.get("default_backbone_weights", {})
    if species in species_weights:
        return {str(k): float(v) for k, v in species_weights[species].items()}
    return {str(k): float(v) for k, v in default_weights.items()}


def build_species_features(bundle, recipe):
    variants = recipe.get("tta_variants", ["identity", "hflip"])
    backbones = get_species_backbones(recipe, bundle["species"])
    backbone_weights = get_species_backbone_weights(recipe, bundle["species"])

    if not backbone_weights:
        backbone_weights = {name: 1.0 for name in backbones}

    total = sum(max(0.0, float(backbone_weights.get(name, 0.0))) for name in backbones)
    if total <= 0:
        backbone_weights = {name: 1.0 / len(backbones) for name in backbones}
    else:
        backbone_weights = {name: float(backbone_weights.get(name, 0.0)) / total for name in backbones}

    features_to_concat = []
    for backbone_name in backbones:
        if backbone_name not in bundle["backbones"]:
            raise KeyError(f"Backbone {backbone_name} missing for {bundle['species']}.")

        bb = bundle["backbones"][backbone_name]
        masked = average_tta_bank(bb["masked_bank"], variants)
        original = average_tta_bank(bb["original_bank"], variants)
        fused = fuse_feature_pair(
            masked,
            original,
            mask_weight=float(recipe.get("mask_weight", 0.70)),
            original_weight=float(recipe.get("original_weight", 0.30)),
            fusion_mode=recipe.get("fusion_mode", "concat"),
        )
        features_to_concat.append(backbone_weights[backbone_name] * fused)

    if len(features_to_concat) == 1:
        return l2_normalize(features_to_concat[0])
    return l2_normalize(np.concatenate(features_to_concat, axis=1))


def threshold_for_species(species, bundle, all_features, recipe):
    overrides = recipe.get("threshold_overrides", {})
    if species in overrides:
        return float(overrides[species]), None

    mode = recipe.get("threshold_mode", "train_tune")
    scale = float(recipe.get("threshold_scale", 1.0))
    base_threshold = float(SPECIES_CONFIG[species]["base_threshold"])

    if mode == "known_good":
        return float(base_threshold * scale), None

    if mode == "train_tune":
        train_rows = bundle["species_rows"][bundle["species_rows"]["split"] == "train"].copy()
        if len(train_rows) == 0:
            return float(base_threshold * scale), []

        train_indices = get_local_indices(train_rows, bundle["local_index_by_global"])
        train_features = all_features[train_indices]
        train_features = query_expansion(
            train_features,
            k=SPECIES_CONFIG[species]["qe_k"],
            alpha=float(recipe.get("qe_alpha", 0.5)),
        )
        tuned_threshold, history = tune_threshold_on_train(
            train_features,
            train_rows["identity"].astype(str).values,
            base_threshold=base_threshold,
        )
        return float(tuned_threshold * scale), history

    raise ValueError(f"Unknown threshold_mode: {mode}")


def build_candidate_submission(recipe, species_bundles):
    parts = []
    report_rows = []

    for species, bundle in species_bundles.items():
        all_features = build_species_features(bundle, recipe)
        threshold, tune_history = threshold_for_species(species, bundle, all_features, recipe)

        test_rows = bundle["species_rows"][bundle["species_rows"]["split"] == "test"].copy()
        test_indices = get_local_indices(test_rows, bundle["local_index_by_global"])
        test_features = all_features[test_indices]
        test_features = query_expansion(
            test_features,
            k=SPECIES_CONFIG[species]["qe_k"],
            alpha=float(recipe.get("qe_alpha", 0.5)),
        )

        part = make_submission_part(
            features=test_features,
            image_ids=test_rows["image_id"].values,
            species=species,
            threshold=threshold,
        )
        parts.append(part)

        row = {
            "profile": recipe["name"],
            "species": species,
            "backbones": "+".join(get_species_backbones(recipe, species)),
            "backbone_weights": json.dumps(get_species_backbone_weights(recipe, species)),
            "threshold": float(threshold),
            "threshold_mode": recipe.get("threshold_mode", "train_tune"),
            "threshold_scale": float(recipe.get("threshold_scale", 1.0)),
            "fusion_mode": recipe.get("fusion_mode", "concat"),
            "mask_weight": float(recipe.get("mask_weight", 0.70)),
            "original_weight": float(recipe.get("original_weight", 0.30)),
            "qe_alpha": float(recipe.get("qe_alpha", 0.5)),
            "tta_variants": "+".join(recipe.get("tta_variants", ["identity", "hflip"])),
            "mask_coverage": float(bundle["species_rows"]["mask_found"].mean()) if "mask_found" in bundle["species_rows"] else 0.0,
        }
        row.update(cluster_shape(part))
        if tune_history:
            best = sorted(tune_history, key=lambda x: x[1], reverse=True)[0]
            row["train_best_ari"] = float(best[1])
            row["train_best_n_clusters"] = int(best[2])
        else:
            row["train_best_ari"] = np.nan
            row["train_best_n_clusters"] = np.nan
        report_rows.append(row)

    predictions = pd.concat(parts, ignore_index=True)
    submission = sample_submission[["image_id"]].merge(predictions, on="image_id", how="left")
    assert len(submission) == len(sample_submission), "Submission length mismatch."
    assert submission["cluster"].notna().all(), "Some test image_ids did not receive a cluster."
    return submission, pd.DataFrame(report_rows)


def recipe_needed_backbones(candidate_recipes, species_list):
    needed = {species: set() for species in species_list}
    for recipe in candidate_recipes:
        for species in species_list:
            for backbone in get_species_backbones(recipe, species):
                needed[species].add(backbone)
    return needed


def species_proxy_score(row):
    species = row["species"]
    score = 0.0
    if pd.notna(row.get("train_best_ari", np.nan)):
        score += float(row["train_best_ari"])

    n_clusters = float(row["n_clusters"])
    max_cluster_size = float(row["max_cluster_size"])
    singletons = float(row["singletons"])

    regime_penalty = 0.0
    for anchor in ANCHOR_REGIMES.values():
        weight = float(anchor.get("weight", 0.5))
        target = float(anchor["target_clusters"][species])
        max_ok = float(anchor["max_cluster_ok"][species])
        regime_penalty += weight * (
            0.08 * abs(n_clusters - target) / max(target, 1.0)
            + 0.04 * max(0.0, max_cluster_size - max_ok) / max(max_ok, 1.0)
        )

    if species == "TexasHornedLizards":
        blended_texas_target = sum(
            float(anchor["weight"]) * float(anchor["target_clusters"]["TexasHornedLizards"])
            for anchor in ANCHOR_REGIMES.values()
        )
        blended_texas_max = sum(
            float(anchor["weight"]) * float(anchor["max_cluster_ok"]["TexasHornedLizards"])
            for anchor in ANCHOR_REGIMES.values()
        )
        regime_penalty += 0.04 * abs(n_clusters - blended_texas_target) / max(blended_texas_target, 1.0)
        regime_penalty += 0.02 * max(0.0, max_cluster_size - blended_texas_max) / max(blended_texas_max, 1.0)
        if singletons < 5:
            regime_penalty += 0.02

    score -= regime_penalty
    return float(score)


def summarize_profiles(candidate_report):
    rows = []
    for profile, group in candidate_report.groupby("profile"):
        proxy = float(sum(species_proxy_score(row) for _, row in group.iterrows()))
        row = {
            "profile": profile,
            "proxy_score": proxy,
            "mean_train_best_ari": float(group["train_best_ari"].dropna().mean()) if group["train_best_ari"].notna().any() else np.nan,
        }
        for species in SPECIES_ORDER:
            sub = group[group["species"] == species]
            if len(sub) == 0:
                continue
            row[f"{species}_n_clusters"] = int(sub["n_clusters"].iloc[0])
            row[f"{species}_max_cluster_size"] = int(sub["max_cluster_size"].iloc[0])
        rows.append(row)
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["proxy_score", "mean_train_best_ari"], ascending=[False, False]).reset_index(drop=True)


def build_specieswise_hybrid(candidate_report, written_paths, metadata, sample_submission):
    if len(candidate_report) == 0:
        return None, pd.DataFrame(), None

    test_meta = metadata[metadata["split"] == "test"][["image_id", "dataset"]].copy()
    profile_cache = {}
    parts = []
    choice_rows = []

    for species in SPECIES_ORDER:
        group = candidate_report[candidate_report["species"] == species].copy()
        if len(group) == 0:
            continue

        group["species_proxy_score"] = [
            species_proxy_score(row) for _, row in group.iterrows()
        ]
        sort_cols = ["species_proxy_score"]
        ascending = [False]
        if "train_best_ari" in group.columns:
            sort_cols.append("train_best_ari")
            ascending.append(False)
        group = group.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)
        best = group.iloc[0]
        profile = str(best["profile"])

        if profile not in profile_cache:
            profile_cache[profile] = pd.read_csv(written_paths[profile])

        species_ids = set(
            test_meta[test_meta["dataset"] == species]["image_id"].astype(int).tolist()
        )
        part = profile_cache[profile][
            profile_cache[profile]["image_id"].astype(int).isin(species_ids)
        ].copy()
        assert len(part) == len(species_ids), f"Hybrid extraction failed for {species}"
        parts.append(part)

        row = best.to_dict()
        row["species_proxy_score"] = float(best["species_proxy_score"])
        choice_rows.append(row)

    if not parts:
        return None, pd.DataFrame(), None

    predictions = pd.concat(parts, ignore_index=True)
    hybrid_submission = sample_submission[["image_id"]].merge(
        predictions, on="image_id", how="left"
    )
    assert len(hybrid_submission) == len(sample_submission), "Hybrid submission length mismatch."
    assert hybrid_submission["cluster"].notna().all(), "Hybrid submission has missing clusters."

    choice_df = pd.DataFrame(choice_rows)
    hybrid_summary = {
        "profile": "specieswise_hybrid",
        "proxy_score": float(choice_df["species_proxy_score"].sum()),
        "mean_train_best_ari": float(choice_df["train_best_ari"].dropna().mean()) if choice_df["train_best_ari"].notna().any() else np.nan,
    }
    for species in SPECIES_ORDER:
        sub = choice_df[choice_df["species"] == species]
        if len(sub) == 0:
            continue
        hybrid_summary[f"{species}_source_profile"] = str(sub["profile"].iloc[0])
        hybrid_summary[f"{species}_n_clusters"] = int(sub["n_clusters"].iloc[0])
        hybrid_summary[f"{species}_max_cluster_size"] = int(sub["max_cluster_size"].iloc[0])
    return hybrid_submission, choice_df, hybrid_summary
        

In [ ]:
# =========================
# Main execution
# =========================

CANDIDATE_RECIPES = [
    {
        "name": "q00_anchor_previous_0274_style",
        "species_backbones": {
            "LynxID2025": ["miewid"],
            "SalamanderID2025": ["mega_l384_oldnorm"],
            "SeaTurtleID2022": ["mega_l384_oldnorm"],
            "TexasHornedLizards": ["miewid"],
        },
        "threshold_mode": "known_good",
        "fusion_mode": "concat",
        "mask_weight": 0.70,
        "original_weight": 0.30,
        "qe_alpha": 0.5,
        "tta_variants": ["identity", "hflip", "contrast"],
    },
    {
        "name": "q00b_dualfoundation_lbshape_bridge",
        "species_backbones": {
            "LynxID2025": ["miewid", "mega_l384_oldnorm"],
            "SalamanderID2025": ["mega_l384_oldnorm", "miewid"],
            "SeaTurtleID2022": ["mega_l384_oldnorm", "miewid"],
            "TexasHornedLizards": ["miewid", "mega_l384_oldnorm"],
        },
        "species_backbone_weights": {
            "LynxID2025": {"miewid": 0.72, "mega_l384_oldnorm": 0.28},
            "SalamanderID2025": {"mega_l384_oldnorm": 0.78, "miewid": 0.22},
            "SeaTurtleID2022": {"mega_l384_oldnorm": 0.72, "miewid": 0.28},
            "TexasHornedLizards": {"miewid": 0.84, "mega_l384_oldnorm": 0.16},
        },
        "threshold_mode": "train_tune",
        "threshold_scale": 0.99,
        "threshold_overrides": {"TexasHornedLizards": 0.31},
        "fusion_mode": "concat",
        "mask_weight": 0.72,
        "original_weight": 0.28,
        "qe_alpha": 0.5,
        "tta_variants": ["identity", "hflip", "contrast"],
    },
    {
        "name": "q01_dualfoundation_species_mix_modern",
        "species_backbones": {
            "LynxID2025": ["miewid", "mega_l384"],
            "SalamanderID2025": ["mega_l384", "miewid"],
            "SeaTurtleID2022": ["mega_l384", "miewid"],
            "TexasHornedLizards": ["miewid", "mega_l384"],
        },
        "species_backbone_weights": {
            "LynxID2025": {"miewid": 0.68, "mega_l384": 0.32},
            "SalamanderID2025": {"mega_l384": 0.78, "miewid": 0.22},
            "SeaTurtleID2022": {"mega_l384": 0.72, "miewid": 0.28},
            "TexasHornedLizards": {"miewid": 0.80, "mega_l384": 0.20},
        },
        "threshold_mode": "train_tune",
        "threshold_overrides": {"TexasHornedLizards": 0.30},
        "fusion_mode": "concat",
        "mask_weight": 0.70,
        "original_weight": 0.30,
        "qe_alpha": 0.5,
        "tta_variants": ["identity", "hflip", "contrast"],
    },
    {
        "name": "q02_dualfoundation_species_mix_oldnorm",
        "species_backbones": {
            "LynxID2025": ["miewid", "mega_l384_oldnorm"],
            "SalamanderID2025": ["mega_l384_oldnorm", "miewid"],
            "SeaTurtleID2022": ["mega_l384_oldnorm", "miewid"],
            "TexasHornedLizards": ["miewid", "mega_l384_oldnorm"],
        },
        "species_backbone_weights": {
            "LynxID2025": {"miewid": 0.68, "mega_l384_oldnorm": 0.32},
            "SalamanderID2025": {"mega_l384_oldnorm": 0.80, "miewid": 0.20},
            "SeaTurtleID2022": {"mega_l384_oldnorm": 0.74, "miewid": 0.26},
            "TexasHornedLizards": {"miewid": 0.82, "mega_l384_oldnorm": 0.18},
        },
        "threshold_mode": "train_tune",
        "threshold_overrides": {"TexasHornedLizards": 0.30},
        "fusion_mode": "concat",
        "mask_weight": 0.70,
        "original_weight": 0.30,
        "qe_alpha": 0.5,
        "tta_variants": ["identity", "hflip", "contrast"],
    },
    {
        "name": "q03_trifoundation_species_mix",
        "species_backbones": {
            "LynxID2025": ["miewid", "mega_l384_oldnorm", "mega_l384"],
            "SalamanderID2025": ["mega_l384", "mega_l384_oldnorm", "miewid"],
            "SeaTurtleID2022": ["mega_l384", "mega_l384_oldnorm", "miewid"],
            "TexasHornedLizards": ["miewid", "mega_l384_oldnorm", "mega_l384"],
        },
        "species_backbone_weights": {
            "LynxID2025": {"miewid": 0.56, "mega_l384_oldnorm": 0.24, "mega_l384": 0.20},
            "SalamanderID2025": {"mega_l384": 0.48, "mega_l384_oldnorm": 0.40, "miewid": 0.12},
            "SeaTurtleID2022": {"mega_l384": 0.46, "mega_l384_oldnorm": 0.39, "miewid": 0.15},
            "TexasHornedLizards": {"miewid": 0.60, "mega_l384_oldnorm": 0.22, "mega_l384": 0.18},
        },
        "threshold_mode": "train_tune",
        "threshold_overrides": {"TexasHornedLizards": 0.30},
        "fusion_mode": "concat",
        "mask_weight": 0.70,
        "original_weight": 0.30,
        "qe_alpha": 0.5,
        "tta_variants": ["identity", "hflip", "contrast"],
    },
    {
        "name": "q04_dualfoundation_mask75_modern",
        "species_backbones": {
            "LynxID2025": ["miewid", "mega_l384"],
            "SalamanderID2025": ["mega_l384", "miewid"],
            "SeaTurtleID2022": ["mega_l384", "miewid"],
            "TexasHornedLizards": ["miewid", "mega_l384"],
        },
        "species_backbone_weights": {
            "LynxID2025": {"miewid": 0.70, "mega_l384": 0.30},
            "SalamanderID2025": {"mega_l384": 0.80, "miewid": 0.20},
            "SeaTurtleID2022": {"mega_l384": 0.73, "miewid": 0.27},
            "TexasHornedLizards": {"miewid": 0.82, "mega_l384": 0.18},
        },
        "threshold_mode": "train_tune",
        "threshold_scale": 1.02,
        "threshold_overrides": {"TexasHornedLizards": 0.31},
        "fusion_mode": "concat",
        "mask_weight": 0.75,
        "original_weight": 0.25,
        "qe_alpha": 0.5,
        "tta_variants": ["identity", "hflip", "contrast"],
    },
    {
        "name": "q05_dualfoundation_mask65_oldnorm",
        "species_backbones": {
            "LynxID2025": ["miewid", "mega_l384_oldnorm"],
            "SalamanderID2025": ["mega_l384_oldnorm", "miewid"],
            "SeaTurtleID2022": ["mega_l384_oldnorm", "miewid"],
            "TexasHornedLizards": ["miewid", "mega_l384_oldnorm"],
        },
        "species_backbone_weights": {
            "LynxID2025": {"miewid": 0.66, "mega_l384_oldnorm": 0.34},
            "SalamanderID2025": {"mega_l384_oldnorm": 0.78, "miewid": 0.22},
            "SeaTurtleID2022": {"mega_l384_oldnorm": 0.72, "miewid": 0.28},
            "TexasHornedLizards": {"miewid": 0.80, "mega_l384_oldnorm": 0.20},
        },
        "threshold_mode": "train_tune",
        "threshold_scale": 0.98,
        "threshold_overrides": {"TexasHornedLizards": 0.29},
        "fusion_mode": "concat",
        "mask_weight": 0.65,
        "original_weight": 0.35,
        "qe_alpha": 0.5,
        "tta_variants": ["identity", "hflip", "contrast"],
    },
    {
        "name": "q06_dualnorm_species_mix",
        "species_backbones": {
            "LynxID2025": ["miewid", "mega_l384_oldnorm"],
            "SalamanderID2025": ["mega_l384", "miewid"],
            "SeaTurtleID2022": ["mega_l384", "miewid"],
            "TexasHornedLizards": ["miewid", "mega_l384_oldnorm"],
        },
        "species_backbone_weights": {
            "LynxID2025": {"miewid": 0.70, "mega_l384_oldnorm": 0.30},
            "SalamanderID2025": {"mega_l384": 0.80, "miewid": 0.20},
            "SeaTurtleID2022": {"mega_l384": 0.73, "miewid": 0.27},
            "TexasHornedLizards": {"miewid": 0.82, "mega_l384_oldnorm": 0.18},
        },
        "threshold_mode": "train_tune",
        "threshold_overrides": {"TexasHornedLizards": 0.30},
        "fusion_mode": "concat",
        "mask_weight": 0.70,
        "original_weight": 0.30,
        "qe_alpha": 0.5,
        "tta_variants": ["identity", "hflip", "contrast"],
    },
]

species_list = list(metadata[metadata["split"] == "test"]["dataset"].unique())
needed = recipe_needed_backbones(CANDIDATE_RECIPES, species_list)

print("Species list:", species_list)
print("Needed backbones:")
for species in species_list:
    print(" -", species, ":", sorted(needed[species]))

species_bundles = {}
for species in species_list:
    print("\n" + "=" * 80)
    print("Preparing species:", species)

    species_rows = metadata[metadata["dataset"] == species].copy().reset_index(drop=True)
    masked_paths = species_rows["masked_path"].tolist()
    original_paths = species_rows["abs_path"].tolist()
    local_index_by_global = {
        int(global_idx): local_idx
        for local_idx, global_idx in enumerate(species_rows["row_idx"].tolist())
    }

    species_bundles[species] = {
        "species": species,
        "species_rows": species_rows,
        "local_index_by_global": local_index_by_global,
        "backbones": {},
    }

    for backbone_name in sorted(needed[species]):
        print("\n" + "-" * 70)
        print(f"Extracting {species} with backbone: {backbone_name}")
        try:
            model = get_model(backbone_name)
        except Exception as e:
            print(f"[skip] Could not load {backbone_name}: {repr(e)}")
            continue

        try:
            variants_to_extract = sorted(set(
                variant
                for recipe in CANDIDATE_RECIPES
                if backbone_name in get_species_backbones(recipe, species)
                for variant in recipe.get("tta_variants", ["identity", "hflip"])
            ))
            print("TTA variants:", variants_to_extract)

            masked_bank = extract_tta_bank(
                model=model,
                paths=masked_paths,
                backbone_name=backbone_name,
                cache_prefix=f"{species}_{backbone_name}_masked_{VERSION}",
                variants=variants_to_extract,
            )
            original_bank = extract_tta_bank(
                model=model,
                paths=original_paths,
                backbone_name=backbone_name,
                cache_prefix=f"{species}_{backbone_name}_original_{VERSION}",
                variants=variants_to_extract,
            )
            species_bundles[species]["backbones"][backbone_name] = {
                "masked_bank": masked_bank,
                "original_bank": original_bank,
            }
        except Exception as e:
            print(f"[skip] Feature extraction failed for {species} / {backbone_name}: {repr(e)}")

        del model
        cleanup_memory(f"after {species} / {backbone_name}")

all_reports = []
written_paths = {}
skipped_recipes = []

for recipe in CANDIDATE_RECIPES:
    print("\n" + "=" * 80)
    print("Building candidate:", recipe["name"])

    missing = []
    for species in species_list:
        for backbone_name in get_species_backbones(recipe, species):
            if backbone_name not in species_bundles[species]["backbones"]:
                missing.append((species, backbone_name))

    if missing:
        print("[skip] Missing extracted backbones:", missing)
        skipped_recipes.append({"profile": recipe["name"], "reason": str(missing)})
        continue

    submission, report = build_candidate_submission(recipe, species_bundles)
    output_path = SUB_DIR / f"submission_{recipe['name']}.csv"
    submission.to_csv(output_path, index=False)
    written_paths[recipe["name"]] = output_path
    all_reports.append(report)
    print("Wrote:", output_path)
    display(report)

if all_reports:
    candidate_report = pd.concat(all_reports, ignore_index=True)
else:
    candidate_report = pd.DataFrame()

profile_summary = summarize_profiles(candidate_report) if len(candidate_report) else pd.DataFrame()
species_hybrid_submission, species_hybrid_choices, species_hybrid_summary = build_specieswise_hybrid(
    candidate_report=candidate_report,
    written_paths=written_paths,
    metadata=metadata,
    sample_submission=sample_submission,
) if len(candidate_report) else (None, pd.DataFrame(), None)

if species_hybrid_summary is not None:
    profile_summary = pd.concat(
        [profile_summary, pd.DataFrame([species_hybrid_summary])],
        ignore_index=True,
    ).sort_values(["proxy_score", "mean_train_best_ari"], ascending=[False, False]).reset_index(drop=True)

report_path = WORK_DIR / "dualfoundation_candidate_report_v20260425.csv"
profile_path = WORK_DIR / "dualfoundation_profile_summary_v20260425.csv"
skipped_path = WORK_DIR / "dualfoundation_skipped_profiles_v20260425.csv"
selected_meta_path = WORK_DIR / "dualfoundation_selected_profile_v20260425.json"
species_hybrid_report_path = WORK_DIR / "dualfoundation_species_hybrid_report_v20260425.csv"
species_hybrid_submission_path = WORK_DIR / "submission_species_hybrid_v20260425.csv"

candidate_report.to_csv(report_path, index=False)
profile_summary.to_csv(profile_path, index=False)
pd.DataFrame(skipped_recipes).to_csv(skipped_path, index=False)
species_hybrid_choices.to_csv(species_hybrid_report_path, index=False)
if species_hybrid_submission is not None:
    species_hybrid_submission.to_csv(species_hybrid_submission_path, index=False)
    species_hybrid_submission.to_csv(SUB_DIR / "submission_species_hybrid_v20260425.csv", index=False)

chosen_profile = None
if len(profile_summary):
    if DEFAULT_PROFILE_HINT and DEFAULT_PROFILE_HINT in set(profile_summary["profile"]):
        top_proxy = float(profile_summary.iloc[0]["proxy_score"])
        hint_proxy = float(profile_summary[profile_summary["profile"] == DEFAULT_PROFILE_HINT]["proxy_score"].iloc[0])
        if hint_proxy >= top_proxy - 0.02:
            chosen_profile = DEFAULT_PROFILE_HINT
    if chosen_profile is None:
        chosen_profile = str(profile_summary.iloc[0]["profile"])

if chosen_profile is not None:
    default_path = WORK_DIR / "submission.csv"
    named_selected_path = WORK_DIR / f"submission_selected_{chosen_profile}.csv"
    if chosen_profile == "specieswise_hybrid":
        selected_path = species_hybrid_submission_path
        selected_df = species_hybrid_submission.copy()
    else:
        selected_path = written_paths[chosen_profile]
        selected_df = pd.read_csv(selected_path)
    selected_df.to_csv(default_path, index=False)
    selected_df.to_csv(named_selected_path, index=False)

    selected_summary = profile_summary[profile_summary["profile"] == chosen_profile].iloc[0].to_dict()
    selected_meta = {
        "chosen_profile": chosen_profile,
        "selected_submission": str(selected_path),
        "named_selected_submission": str(named_selected_path),
        "default_submission": str(default_path),
        "summary": selected_summary,
    }
    if chosen_profile == "specieswise_hybrid":
        selected_meta["species_hybrid_choices"] = species_hybrid_choices.to_dict(orient="records")
    Path(selected_meta_path).write_text(json.dumps(selected_meta, indent=2), encoding="utf-8")
    print("Default submission.csv written from:", chosen_profile)
    print("Named selected submission:", named_selected_path)
else:
    print("WARNING: No candidate profile was available.")

print("\nCandidate files:")
for name, path in written_paths.items():
    print(f" - {name}: {path}")
print("\nReport:", report_path)
print("Profile summary:", profile_path)
print("Skipped:", skipped_path)
print("Species hybrid report:", species_hybrid_report_path)
print("Species hybrid submission:", species_hybrid_submission_path)
print("Selection meta:", selected_meta_path)
print("Default submission:", WORK_DIR / "submission.csv")
print("Chosen profile:", chosen_profile)

display(profile_summary)
display(candidate_report)
if len(species_hybrid_choices):
    display(species_hybrid_choices)
        